# 10 — RG Flow & Gauge Unification

**The 24-cell root system and the emergence of the Standard Model.**

SDGFT derives the SM gauge group from geometry:

$$\text{24-cell}(D_4) \;\supset\; SU(3)_C \times SU(2)_L \times U(1)_Y$$

This notebook covers:
1. **24-cell geometry**: 24 vertices → D₄ root system
2. **SM decomposition**: 12 gauge bosons + 8 matter pairs
3. **Triality**: S₃ symmetry permuting representations
4. **1-loop RG**: gauge coupling running M_Z → M_Pl
5. **SDGFT boundary**: α₁/α₂ = 5/24 = Δ,  sin²θ_W = 1/9

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import math

from sdgft_ml.physics import constants as C
from sdgft_ml.physics import dimension as D
from sdgft_ml.physics import gauge_groups as gg
from sdgft_ml.physics import rg_running as rg

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
print("Imports OK")
print(f"  Δ = {C.DELTA:.10f}  (5/24)")
print(f"  δ = {C.DELTA_G:.10f}  (1/24)")

---
## 1  The 24-Cell: Vertices and Root System

The 24-cell is the unique self-dual regular 4-polytope.
Its 24 vertices form the D₄ root system in 4D.

In [ ]:
# Get 24-cell vertices (module constant, not a function)
vertices = gg.VERTICES_24CELL
print(f"24-cell: {len(vertices)} vertices")
print(f"\nFirst 8 (permutations of ±2,0,0,0):")
for v in vertices[:8]:
    print(f"  ({v[0]:+d}, {v[1]:+d}, {v[2]:+d}, {v[3]:+d})")
print(f"\nNext vertices (±1,±1,±1,±1):")
for v in vertices[8:12]:
    print(f"  ({v[0]:+d}, {v[1]:+d}, {v[2]:+d}, {v[3]:+d})")
print(f"  ... ({len(vertices)} vertices total)")

# Verify norms
norms = [math.sqrt(sum(x**2 for x in v)) for v in vertices]
print(f"\nVertex norms: {set(f'{n:.3f}' for n in norms)}")

In [ ]:
# 3D projection of 24-cell vertices
verts = np.array(vertices)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

# Project 4D → 3D via stereographic-like projection
# Use first 3 coordinates, colour by 4th
x, y, z, w = verts[:, 0], verts[:, 1], verts[:, 2], verts[:, 3]
colors = plt.cm.coolwarm((w - w.min()) / (w.max() - w.min() + 1e-10))

ax.scatter(x, y, z, c=colors, s=100, edgecolors="k", linewidth=0.5)

# Draw edges (vertices within distance √2 of each other)
for i in range(len(verts)):
    for j in range(i+1, len(verts)):
        d = math.sqrt(sum((verts[i][k] - verts[j][k])**2 for k in range(4)))
        if abs(d - math.sqrt(2)) < 0.01:
            ax.plot([x[i], x[j]], [y[i], y[j]], [z[i], z[j]],
                    "k-", alpha=0.15, lw=0.5)

ax.set_xlabel("x₁"); ax.set_ylabel("x₂"); ax.set_zlabel("x₃")
ax.set_title("24-Cell Vertices (3D projection, colour = x₄)")
plt.tight_layout()
plt.show()

---
## 2  D₄ Root System and Cartan Matrix

The D₄ root system has rank 4, 24 roots, and the characteristic
trivalent Dynkin diagram.

In [ ]:
# D4 roots (module constant)
roots = gg.D4_ROOTS
print(f"D₄ root system: {len(roots)} roots")

# Verify isomorphism with 24-cell
iso = gg.verify_24cell_d4_isomorphism()
print(f"\n24-cell ≅ D₄ isomorphism test:")
for k, v in iso.items():
    if isinstance(v, bool):
        print(f"  {k}: {'✓' if v else '✗'}")
    else:
        print(f"  {k}: {v}")

In [ ]:
# Cartan matrix (module constant)
A = gg.D4_CARTAN_MATRIX
print("D₄ Cartan matrix:")
for row in A:
    print("  [" + ", ".join(f"{x:+d}" for x in row) + "]")

# Visualise as heatmap
A_arr = np.array(A)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(A_arr, cmap="RdBu_r", vmin=-2, vmax=2)
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(["α₁", "α₂", "α₃", "α₄"])
ax.set_yticklabels(["α₁", "α₂", "α₃", "α₄"])
ax.set_title("D₄ Cartan Matrix")
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{A_arr[i][j]:+d}", ha="center", va="center", fontsize=14)
plt.colorbar(im)
plt.tight_layout()
plt.show()

---
## 3  Standard Model Decomposition

The 24 roots decompose as:
$$D_4 \;\to\; \underbrace{A_2}_{SU(3)} \;\oplus\; \underbrace{A_1}_{SU(2)} \;\oplus\; \underbrace{U(1)}_Y$$

giving 8 gluons + 3 weak bosons + 1 hypercharge = 12 gauge bosons
and 12 matter-sector roots.

In [ ]:
# Decompose D4 → SM (returns GaugeDecomposition NamedTuple)
decomp = gg.decompose_d4_to_sm()

print("D₄ → Standard Model decomposition:")
print(f"  Gauge bosons: {decomp.n_gauge_bosons}")
print(f"  Coset roots:  {decomp.n_coset}")
print(f"  Total roots:  {decomp.n_total_roots}")

print(f"\nGauge sector:")
print(f"  SU(3) [A₂]: dim = {decomp.a2_dim}  (rank {decomp.a2_cartan_rank}, {len(decomp.a2_roots)} roots)")
print(f"  SU(2) [A₁]: dim = {decomp.a1_dim}  (rank {decomp.a1_cartan_rank}, {len(decomp.a1_roots)} roots)")
print(f"  U(1):        dim = {decomp.u1_dim}  (rank {decomp.u1_cartan_rank})")

print(f"\nMatter sector:")
print(f"  {decomp.n_coset} coset roots = {decomp.n_coset // 2} coset pairs")
print(f"  → quarks + leptons in fundamental representations")

In [ ]:
# δ = 1/|roots| = 1/24
print(f"δ = 1/|D₄ roots| = 1/{len(roots)} = {1/len(roots):.10f}")
print(f"δ (axiom) = {C.DELTA_G:.10f}")
print(f"Match: {abs(1/len(roots) - float(C.DELTA_G)) < 1e-12}")

---
## 4  D₄ Triality

D₄ has a unique S₃ outer automorphism group (triality)
that permutes the vector and two spinor representations.

$$|\text{Aut}(D_4)| = |W(D_4)| \cdot |S_3| = 192 \times 6 = 1152$$

In [ ]:
# Verify triality
tri = gg.verify_triality()
print("D₄ Triality verification:")
for k, v in tri.items():
    if isinstance(v, bool):
        print(f"  {k}: {'✓' if v else '✗'}")
    elif isinstance(v, (int, float)):
        print(f"  {k}: {v}")
    else:
        print(f"  {k}: {v}")

In [ ]:
# Coset structure
pairs = gg.coset_pairs()
print(f"\nCoset pairs (root → −root):")
for i, (r, mr) in enumerate(pairs[:6]):
    r_str = "(" + ", ".join(f"{x:+.1f}" for x in r) + ")"
    mr_str = "(" + ", ".join(f"{x:+.1f}" for x in mr) + ")"
    print(f"  Pair {i+1}: {r_str} ↔ {mr_str}")
print(f"  ... {len(pairs)} pairs total")

---
## 5  SM Gauge Coupling Running (1-Loop)

Standard 1-loop RG equations with SM particle content:

$$\alpha_i^{-1}(\mu) = \alpha_i^{-1}(M_Z) - \frac{b_i}{2\pi}\ln\frac{\mu}{M_Z}$$

In [ ]:
# Full RG trajectory
trajectory = rg.rg_trajectory()

energies = [p["scale_gev"] for p in trajectory]
ia1 = [p["inv_alpha_1"] for p in trajectory]
ia2 = [p["inv_alpha_2"] for p in trajectory]
ia3 = [p["inv_alpha_3"] for p in trajectory]
sin2tw = [p["sin2_theta_w"] for p in trajectory]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: inverse couplings
ax1.semilogx(energies, ia1, "r-", lw=2, label="α₁⁻¹ (U(1))")
ax1.semilogx(energies, ia2, "b-", lw=2, label="α₂⁻¹ (SU(2))")
ax1.semilogx(energies, ia3, "g-", lw=2, label="α₃⁻¹ (SU(3))")

# GUT scale — find_unification_scale returns (t_gut, m_gut_gev) tuple
gut = rg.find_unification_scale()
if gut:
    m_gut_gev = gut[1]
    ax1.axvline(m_gut_gev, color="purple", ls=":", lw=1.5,
                label=f"GUT ~ {m_gut_gev:.1e} GeV")

ax1.axvline(rg.M_PL, color="gray", ls=":", alpha=0.5, label="M_Pl")
ax1.set_xlabel("μ  (GeV)"); ax1.set_ylabel("1/α_i")
ax1.set_title("SM Gauge Coupling Running (1-loop)")
ax1.legend(fontsize=9); ax1.grid(alpha=0.3)
ax1.set_ylim(0, 70)

# Right: sin²θ_W
ax2.semilogx(energies, sin2tw, "m-", lw=2)
ax2.axhline(1/9, color="red", ls="--", lw=1.5, label="SDGFT: 1/9")
ax2.axhline(rg.SIN2_THETA_W_MZ, color="blue", ls=":", label=f"obs: {rg.SIN2_THETA_W_MZ}")
ax2.set_xlabel("μ  (GeV)"); ax2.set_ylabel("sin²θ_W")
ax2.set_title("Weinberg angle running")
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Values at M_Pl
pl = rg.run_to_scale(math.log(rg.M_PL / rg.M_Z))
print(f"\nAt M_Pl = {rg.M_PL:.2e} GeV:")
print(f"  α₁⁻¹ = {pl['inv_alpha_1']:.4f}")
print(f"  α₂⁻¹ = {pl['inv_alpha_2']:.4f}")
print(f"  α₃⁻¹ = {pl['inv_alpha_3']:.4f}")
print(f"  sin²θ_W = {pl['sin2_theta_w']:.6f}")
print(f"  α₁/α₂ = {pl['inv_alpha_2']/pl['inv_alpha_1']:.6f}")

---
## 6  SDGFT Boundary Condition at M_Pl

SDGFT predicts that at the Planck scale:

$$\sin^2\theta_W(M_{\text{Pl}}) = \frac{1}{9}, \qquad \frac{\alpha_1}{\alpha_2} = \frac{5}{24} = \Delta$$

The SM 1-loop running gives sin²θ_W(M_Pl) ≈ 0.47, not 1/9.
This discrepancy is a **prediction**: new physics must appear between M_Z and M_Pl
to bend the trajectory.

In [ ]:
# Test the SDGFT boundary condition
sin2_pl = pl["sin2_theta_w"]
sin2_target = 1.0 / 9.0
print("SDGFT Planck-scale boundary condition:")
print(f"  sin²θ_W(M_Pl) SM run = {sin2_pl:.6f}")
print(f"  sin²θ_W(M_Pl) SDGFT  = {sin2_target:.6f}  (1/9)")
print(f"  Discrepancy:          {abs(sin2_pl - sin2_target):.6f}")
print(f"\n  → Requires BSM threshold corrections between 10¹⁶ and 10¹⁹ GeV.")

# α₁/α₂ ratio
a1_over_a2 = pl["inv_alpha_2"] / pl["inv_alpha_1"]
print(f"\n  α₁/α₂ (SM run to M_Pl) = {a1_over_a2:.6f}")
print(f"  α₁/α₂ (SDGFT target)   = {C.DELTA:.6f}  (Δ = 5/24)")

In [ ]:
# Scan: at what scale does sin²θ_W cross 1/9?
target = 1.0 / 9.0

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogx(energies, sin2tw, "m-", lw=2.5, label="SM 1-loop")
ax.axhline(1/9, color="red", ls="--", lw=2, label="SDGFT: sin²θ_W = 1/9")
ax.axhline(rg.SIN2_THETA_W_MZ, color="blue", ls=":", lw=1.5,
           label=f"M_Z: {rg.SIN2_THETA_W_MZ:.4f}")
ax.axhline(3/8, color="green", ls="--", lw=1.5, alpha=0.5,
           label="SU(5) GUT: 3/8")

ax.fill_between([1e16, 1e19], 0, 0.5, alpha=0.1, color="orange",
                label="BSM threshold region")

ax.set_xlabel("μ  (GeV)")
ax.set_ylabel("sin²θ_W")
ax.set_title("Weinberg Angle: SM Running vs SDGFT Boundary")
ax.set_ylim(0.05, 0.5)
ax.legend(fontsize=9, loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7  Verification: δ = 1/24 Links Geometry to Gauge Theory

The geometric constant δ = 1/24 simultaneously:
1. Counts the 24-cell roots: δ = 1/|roots|
2. Sets the gravitational anomalous dimension: δ_g = 1/24
3. Appears in Δ = 5δ = 5/24 (the spectral dimension shift)

In [ ]:
# The triple role of 1/24
print("═" * 60)
print("The Triple Role of δ = 1/24")
print("═" * 60)
print(f"\n1. Geometry:     |24-cell roots| = {len(roots)}")
print(f"                 δ = 1/{len(roots)} = {1/len(roots):.10f}")
print(f"\n2. Gravity:      δ_g = {C.DELTA_G:.10f}  (anomalous dimension)")
print(f"                 D* = 3 − 5δ_g(1−δ_g) = {D.D_STAR_TREE_F:.10f}")
print(f"\n3. Gauge theory: Δ = 5/24 = {C.DELTA:.10f}")
print(f"                 sin²θ_W(M_Pl) = (3−5Δ)/9 = {(3-5*C.DELTA)/9:.10f}")
print(f"                 = 1/9 × (3 − 25/24) = 1/9 × 47/24")
print(f"\nAll from one number: 24 — the kiss number of the 24-cell.")
print("═" * 60)

---
## Summary

| Result | SDGFT | Status |
|--------|-------|--------|
| 24-cell ≅ D₄ | 24 vertices = 24 roots | ✓ (exact) |
| SM gauge group | D₄ → SU(3)×SU(2)×U(1) | ✓ (algebraic) |
| Triality | |Aut(D₄)| = 1152 | ✓ (computed) |
| δ = 1/24 | Root count = anomalous dim | ✓ (identity) |
| sin²θ_W(M_Pl) = 1/9 | Requires BSM thresholds | Prediction |
| α₁/α₂ = Δ at M_Pl | Tests gauge unification | Prediction |

**The 24-cell is the geometric origin of the Standard Model.**